# One-way ANOVA
Sums of squares and F by hand, scipy's `f_oneway`, the multiple-testing risk, and the Titanic age-by-class case study.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

## 1. Why not several t-tests: the familywise error rate

In [2]:
for m in [1, 3, 10]:
    print(m, 'tests: P(at least one false alarm) =', round(1 - 0.95 ** m, 3))

1 tests: P(at least one false alarm) = 0.05
3 tests: P(at least one false alarm) = 0.143
10 tests: P(at least one false alarm) = 0.401


In [3]:
# The formula assumes independent tests. Pairwise t-tests share groups, so they are not independent.
# Simulate: k groups of 20 values with equal true means; run every pairwise t-test, and ANOVA once.
from itertools import combinations
rng = np.random.default_rng(1)
for k in [3, 5]:
    reps, any_t, anova = 2_000, 0, 0
    for _ in range(reps):
        g = rng.normal(0, 1, size=(k, 20))
        any_t += any(stats.ttest_ind(g[a], g[b]).pvalue <= 0.05 for a, b in combinations(range(k), 2))
        anova += stats.f_oneway(*g).pvalue <= 0.05
    m = k * (k - 1) // 2
    print(f'{k} groups, {m} t-tests: at least one false alarm {any_t / reps:.3f} '
          f'(formula {1 - 0.95 ** m:.3f});  ANOVA false alarm {anova / reps:.3f}')

3 groups, 3 t-tests: at least one false alarm 0.110 (formula 0.143);  ANOVA false alarm 0.050


5 groups, 10 t-tests: at least one false alarm 0.290 (formula 0.401);  ANOVA false alarm 0.050


## 2. ANOVA by hand: marks of three sections

In [4]:
groups = [np.array([4, 5, 6]), np.array([6, 7, 8]), np.array([8, 9, 10])]
all_values = np.concatenate(groups)
grand = all_values.mean()
k, N = len(groups), len(all_values)

sst = ((all_values - grand) ** 2).sum()
ssb = sum(len(g) * (g.mean() - grand) ** 2 for g in groups)     # group means vs grand mean
ssw = sum(((g - g.mean()) ** 2).sum() for g in groups)          # values vs own group mean
print('SST', sst, ' SSB', ssb, ' SSW', ssw, ' SSB + SSW', ssb + ssw)

msb, msw = ssb / (k - 1), ssw / (N - k)
F = msb / msw
print('MSB', msb, ' MSW', msw, ' F', F, ' p', round(stats.f.sf(F, k - 1, N - k), 4))
print('critical value', round(stats.f.ppf(0.95, k - 1, N - k), 2))
print(stats.f_oneway(*groups))

SST 30.0  SSB 24.0  SSW 6.0  SSB + SSW 30.0
MSB 12.0  MSW 1.0  F 12.0  p 0.008
critical value 5.14
F_onewayResult(statistic=np.float64(12.0), pvalue=np.float64(0.008000000000000002))


## 2b. Degrees of freedom and the critical value at two significance levels

In [ ]:
print("df total, between, within:", N - 1, k - 1, N - k)
print(f"critical F(2, 6): alpha 0.05 -> {stats.f.ppf(0.95, 2, 6):.2f}, alpha 0.10 -> {stats.f.ppf(0.90, 2, 6):.2f}")
print(f"p for F = 12: {stats.f.sf(12, 2, 6):.4f}")

## 3. Same means, wider spread

In [5]:
wide = [[1, 5, 9], [3, 7, 11], [5, 9, 13]]
print(stats.f_oneway(*wide))

F_onewayResult(statistic=np.float64(0.75), pvalue=np.float64(0.512))


## 4. Two groups: F equals t squared

In [6]:
print('F =', stats.f_oneway([4, 5, 6], [6, 7, 8]).statistic,
      ' t^2 =', round(stats.ttest_ind([4, 5, 6], [6, 7, 8]).statistic ** 2, 6))

F = 6.0  t^2 = 6.0


## 5. Titanic: age by class

In [7]:
titanic = pd.read_csv('data/titanic_train.csv')
ages = [titanic.loc[titanic['Pclass'] == c, 'Age'].dropna() for c in (1, 2, 3)]
for c, a in zip((1, 2, 3), ages):
    print(f'class {c}: n = {len(a)}, mean {a.mean():.1f}, sd {a.std():.1f}')
print(stats.f_oneway(*ages))
N = sum(len(a) for a in ages)
print('critical value', round(stats.f.ppf(0.95, 2, N - 3), 2))

class 1: n = 186, mean 38.2, sd 14.8
class 2: n = 173, mean 29.9, sd 14.0
class 3: n = 355, mean 25.1, sd 12.5
F_onewayResult(statistic=np.float64(57.44348434067624), pvalue=np.float64(7.487984171957403e-24))
critical value 3.01


In [8]:
# assumption checks
print('Shapiro p:', [f'{stats.shapiro(a).pvalue:.4f}' for a in ages])
print('Levene', stats.levene(*ages))
# tests that drop an assumption
print('Alexander-Govern', stats.alexandergovern(*ages).pvalue)
print('Kruskal-Wallis', stats.kruskal(*ages).pvalue)

Shapiro p: ['0.3643', '0.0056', '0.0000']
Levene LeveneResult(statistic=np.float64(5.620164466250889), pvalue=np.float64(0.003786920599819268))
Alexander-Govern 4.356062111752965e-21
Kruskal-Wallis 1.4288798296552103e-21


In [9]:
# which classes differ? (group 0 = class 1)
print(stats.tukey_hsd(*ages))

Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)      8.356     0.000     5.007    11.704
 (0 - 2)     13.093     0.000    10.223    15.962
 (1 - 0)     -8.356     0.000   -11.704    -5.007
 (1 - 2)      4.737     0.000     1.798     7.676
 (2 - 0)    -13.093     0.000   -15.962   -10.223
 (2 - 1)     -4.737     0.000    -7.676    -1.798

